# Adaptive LASSO on a Dummy Dataset

This notebook walks through a small synthetic regression example using the
`adaptive-lasso` package. The workflow is:

1. Generate a sparse linear dataset.
2. Fit a baseline OLS model.
3. Fit an adaptive-LASSO model.
4. Compare the estimated coefficients.
5. Inspect the selected variables and optional post-selection refit.


For linear regression, adaptive LASSO solves

$$
\hat{\beta}^{\mathrm{adapt}}
=
\arg\min_{\beta}
\left\{
\frac{1}{2n}\lVert y - X\beta \rVert_2^2
+
\lambda \sum_{j=1}^p w_j |\beta_j|
\right\},
$$

with adaptive weights often defined as

$$
w_j = \frac{1}{|\tilde{\beta}_j|^{\gamma} + \varepsilon}.
$$

The main intuition is that predictors with stronger pilot coefficients are
penalized less heavily in the sparse fit.


In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm

from adaptive_lasso import AdaptiveLasso


In [ ]:
rng = np.random.default_rng(123)

n_samples = 250
n_features = 6
feature_names = [f"x{i}" for i in range(1, n_features + 1)]

X = rng.normal(size=(n_samples, n_features))
beta_true = np.array([2.0, -1.5, 0.0, 0.0, 0.75, 0.0])
noise = rng.normal(scale=0.5, size=n_samples)
y = X @ beta_true + noise

design = pd.DataFrame(X, columns=feature_names)
exog = sm.add_constant(design, has_constant="add")

# Leave the intercept unpenalized by assigning a zero penalty to the
# constant column and the same base penalty to the feature columns.
alpha = np.array([0.0] + [0.08] * n_features)

design.head()


In [ ]:
ols_result = sm.OLS(y, exog).fit()

adaptive_result = AdaptiveLasso(y, exog).fit_regularized(
    alpha=alpha,
    L1_wt=1.0,
    adaptive_maxiter=15,
    adaptive_tol=1e-6,
    weight_exponent=1.0,
)


In [ ]:
comparison = pd.DataFrame(
    {
        "true": np.r_[0.0, beta_true],
        "ols": ols_result.params,
        "adaptive_lasso": adaptive_result.params,
    },
    index=exog.columns,
)

comparison


In [ ]:
print("Selected variables:", adaptive_result.selected_variables)
print("Converged:", adaptive_result.converged)
print("Outer iterations:", adaptive_result.fit_history["iterations"])

pd.Series(adaptive_result.effective_alpha, index=exog.columns, name="effective_alpha")


If you want a post-selection OLS fit on the active set, pass `refit=True`.
This estimates sparsity with adaptive LASSO first and then refits the
selected columns without regularization.


In [ ]:
adaptive_refit = AdaptiveLasso(y, exog).fit_regularized(
    alpha=alpha,
    adaptive_maxiter=15,
    adaptive_tol=1e-6,
    refit=True,
)

pd.Series(adaptive_refit.params, index=exog.columns, name="refit_params")


From here you can swap in your own design matrix, adjust the `alpha`
penalty level, or tune `weight_exponent` and `adaptive_maxiter` to explore
different sparsity patterns.
